# SafeBank Assist — experiments E2, E3, E6 (Group 22)
PCAM ZG516 Applied Generative AI · `Qwen/Qwen2.5-1.5B-Instruct` on BANKING77

Runs top to bottom on a **free T4** (Runtime → Change runtime type → T4 GPU), after `safebank_colab.ipynb`'s fixes are on `main`. It is separate from that notebook so the two can be edited and re-run independently; the short setup is repeated here and the data is rebuilt deterministically (seed 42), so both notebooks use the same 385 evaluation rows.

| Experiment | What runs | Generations |
|---|---|---|
| **E2** | 4 prompt strategies × 385 queries, greedy decoding | 1,540 |
| **E3** | 6 decoding configurations × 77 queries (one per intent); sampled configurations once per seed (5 seeds), greedy once | 2,002 |
| **E6** | the six criteria on every condition, baseline vs final, at least 3 analysed failures | none (analysis only) |

**Speed.** Prompts carry all 77 intent labels (about 800 tokens; few-shot about 1,600), so one-at-a-time generation takes hours. `safebank.experiments` batch-generates every first attempt (batch size fixed at 16 and stored in each result) and replays them through the unchanged pipeline, so validation, repair, the confidence gate, tool routing and the safety screen all run as in production. Repair prompts, which are rare, are generated one at a time. `cache_hits` / `cache_misses` below show how much was batched.

**Evidence.** Everything written to `artifacts/results/` is report evidence. Commit it.

## 1. Environment

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || echo "No GPU: switch the runtime to T4"

In [ ]:
# Clone the repo (if it is private, use https://<token>@github.com/... or upload a zip).
import os
REPO_URL = "https://github.com/2025paml539-lab/safebank-assist-group22.git"
BRANCH = "main"
# Until the experiments PR is merged, main has no safebank.experiments.
# Run from the fork branch instead by swapping in these two lines:
# REPO_URL = "https://github.com/rubberduckpanic/safebank-assist-group22.git"
# BRANCH = "2025paml561/s1-s2-report-and-experiments"
if not os.path.exists("safebank-assist-group22"):
    !git clone -q --branch {BRANCH} {REPO_URL}
%cd safebank-assist-group22
!git log -1 --format='%h %s'

In [ ]:
# Only the packages Colab does not already pin. torch is Colab's preinstalled build.
!pip install -q -r requirements-colab.txt

In [ ]:
import sys, json
sys.path.insert(0, "src")
from safebank import config
from safebank.model_loader import device_report
for d in (config.RESULTS_DIR, config.FIGURES_DIR, config.DATA_DIR):
    d.mkdir(parents=True, exist_ok=True)
device_report().as_dict()

## 2. Data
Rebuilds the frozen 385-row evaluation sample (TEST split) and the few-shot bank (TRAIN split). `assert_no_overlap` stops the run if any few-shot example is also an evaluation query, which would contaminate every E2 number.

In [ ]:
from safebank.data_prep import (
    load_banking77, dataset_label_space, build_eval_sample, build_example_bank,
)
from safebank.experiments import assert_no_overlap

train, test = load_banking77()
LABELS = dataset_label_space(train)
eval_rows = build_eval_sample(test, LABELS)            # 5 x 77 = 385, seed 42, TEST split
EXAMPLES = build_example_bank(                         # TRAIN split, never an eval query
    train, LABELS, exclude={r["query"] for r in eval_rows}
)
assert_no_overlap(EXAMPLES, eval_rows)
print(len(LABELS), "intents;", len(eval_rows), "eval rows;", len(EXAMPLES), "few-shot examples")

## 3. Model
Qwen2.5-1.5B-Instruct in fp16. `HFGenerator` resets Qwen's shipped sampling defaults to neutral, so each decoding configuration in E3 means what its label says.

In [ ]:
from safebank.model_loader import load_model_and_tokenizer, HFGenerator

model, tokenizer = load_model_and_tokenizer()
GEN = HFGenerator(model, tokenizer)
BATCH_SIZE = 16        # keep fixed across E2 and E3; stored in every result
print(config.MODEL_ID, "| context window", GEN.context_window, "| batch size", BATCH_SIZE)

## 4. E2 — prompt strategies
Same pipeline, same 385 queries, greedy decoding; only the prompt strategy changes. Few-shot receives the real train-split bank. Qwen's chat template adds a default system prompt to every strategy (report §7), so comparisons stay fair.

In [ ]:
from safebank.experiments import run_e2, report_table, save_condition, save_json
from safebank.evaluate import results_dataframe

e2 = run_e2(GEN, LABELS, eval_rows, EXAMPLES, batch_size=BATCH_SIZE)
for cond in e2:
    save_condition(cond, f"e2_{cond.strategy.value}")
save_json([c.summary() for c in e2], "e2_summary")
results_dataframe(report_table(e2))

In [ ]:
# How much was batched, and what it cost (criterion 5).
for cond in e2:
    m = cond.meta
    print(f"{cond.strategy.value:17s} hits={m['cache_hits']:4d} misses={m['cache_misses']:3d} wall={m['wall_seconds']:7.1f}s")
u = GEN.usage_log
print(f"\n{len(u)} generations | mean prompt tokens {sum(x.prompt_tokens for x in u)/len(u):.0f} | "
      f"mean completion tokens {sum(x.completion_tokens for x in u)/len(u):.0f} | truncated prompts {sum(x.truncated for x in u)}")

## 5. E3 — decoding grid
Runs on the best E2 strategy, over a stratified 77-row subset (one query per intent). Each sampled configuration runs once per seed; **greedy runs once** because it is deterministic, so it has no stability score. `mean_agreement` is the share of seeds that match each query's most common prediction (1.0 = every seed agreed).

In [ ]:
from safebank.experiments import (
    best_condition, select_e3_subset, strategy_kwargs, run_e3, save_grid,
    sample_outputs, variability_examples,
)

BEST = best_condition(e2)
print("E3 runs on the best E2 strategy:", BEST.strategy.value)
subset = select_e3_subset(eval_rows)                   # 1 query per intent = 77 rows
e3 = run_e3(
    GEN, LABELS, subset, BEST.strategy,
    prompt_kwargs=strategy_kwargs(BEST.strategy, EXAMPLES), batch_size=BATCH_SIZE,
)
save_grid(e3)
save_json([g.summary() for g in e3], "e3_summary")
results_dataframe([g.summary() for g in e3])

In [ ]:
# Three sample outputs per configuration, and the queries where seeds disagree.
samples = sample_outputs(e3, k=3)
variability = {g.point.label: variability_examples(g, k=3) for g in e3}
save_json({"samples": samples, "variability": variability}, "e3_samples")
for label, replies in samples.items():
    print("\n==", label)
    for s in replies:
        print("  -", s["query"][:60], "->", s["intent"], "|", (s["response"] or "")[:60])
for label, examples in variability.items():
    if examples:
        print("\nSeeds disagree under", label)
        for ex in examples:
            print("  -", ex["query"][:60], "| gold", ex["gold_intent"], "|", ex["predictions_by_seed"])

## 6. E6 — the six criteria, baseline vs final, failure analysis
The six criteria are in every row above: accuracy and macro-F1, schema validity before and after repair, tool routing, reply safety, latency and attempts, and stability across seeds (E3). Baseline is the zero-shot prompt; final is the best E2 strategy. If they are the same strategy, the comparison is empty by construction: say so in the report.

Failure cases are drawn across different classes (each validation failure kind, a predicted confusable pair, within-theme, cross-theme) so the root-cause table is not three copies of one problem.

In [ ]:
from safebank.experiments import compare_conditions, failure_cases, safety_findings

BASELINE = next(c for c in e2 if c.strategy.value == "zero_shot")
comparison = compare_conditions(BASELINE, BEST)
cases = failure_cases(BEST, n=6)
unsafe = safety_findings(BEST)
save_json(
    {"baseline": BASELINE.strategy.value, "final": BEST.strategy.value,
     "comparison": comparison, "failure_cases": cases, "safety_findings": unsafe},
    "e6_analysis",
)
print(f"{len(cases)} failure cases | {len(unsafe)} replies flagged by the safety screen")
results_dataframe(comparison)

In [ ]:
if cases:
    display(results_dataframe(cases)[["category", "gold_intent", "predicted_intent", "query", "note"]])
unsafe

## 7. Where each result goes in the report

| Result | File in `artifacts/results/` | Report |
|---|---|---|
| E2 comparison table | `e2_summary.json`, `e2_<strategy>.json` | §7, §9 sentence, §1 and §14 headline |
| E4 validity before / after repair | `schema_*` fields in `e2_summary.json` | §10.1, §11 |
| E3 table, samples, stability | `e3_summary.json`, `e3_samples.json`, `e3_<config>_seed<n>.json` | §8 |
| E6 comparison, failures, safety | `e6_analysis.json` | §11, §12.1 |
| Validated, repaired and escalated replies | `attempts_raw` and `escalated` in `e2_<strategy>.json` | Appendix E |

## 8. Save the evidence
Zips `artifacts/results/` for download. Put the contents back into the repo's `artifacts/results/` on the branch and commit them: the report sections are written from these files.

In [ ]:
import shutil
archive = shutil.make_archive("safebank_experiment_results", "zip", str(config.RESULTS_DIR))
print(archive, "|", len(list(config.RESULTS_DIR.glob("*.json"))), "json files")
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Not on Colab: the zip is in the working directory.")
